# Ticket T-105: Feature Engineering Pipeline

This notebook demonstrates, tests, and visualizes the feature engineering pipeline implemented in `src/features.py`.

### Documented Assumption Regarding `trip_distance`:
`trip_distance` is the taximeter-recorded distance of the completed ride. In a real-time production deployment, the taximeter reading would not be available at the start of the trip; a routing engine (e.g. OSRM, Google Maps API) would provide an estimated routing distance instead. Using `trip_distance` as a feature at prediction time is a documented, accepted approximation for this project.

### Acceptance Criteria Verified:
1. **Temporal & Cyclical Features**: `pickup_hour`, `pickup_dayofweek`, `pickup_day`, `sin_hour`, `cos_hour`, `sin_dayofweek`, `cos_dayofweek`, `is_weekend`, `is_rush_hour`, and `is_holiday` (Memorial Day May 30, 2022).
2. **Zone & Spatial Features**: Zone centroids derived from Taxi Zone Shapefile lookup (`dataset/taxi_zone_centroids.csv`); `haversine_distance`, `manhattan_distance`, `haversine_ratio`, `is_same_zone`, `is_jfk`, and `is_newark` flags.
3. **Target Encoding**: High-cardinality zone ID target encoding fitted strictly on training data (`X_train`, `y_train`) using Bayesian smoothing to prevent data leakage.
4. **Unified Serializable Pipeline**: `NYCFeaturePipeline` as a single scikit-learn compatible object serialized to `models/feature_pipeline.pkl`.

In [1]:
import os
import sys
import pickle
from pathlib import Path

# Ensure project root directory is in sys.path when running from notebooks/ directory
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
from src.config import (
    TRAIN_CLEANED_PATH,
    TEST_CLEANED_PATH,
    ALLOWED_FEATURES,
    BANNED_COLUMNS,
    MODELS_DIR
)
from src.features import (
    TemporalFeatureExtractor,
    SpatialZoneFeatureExtractor,
    TargetCategoricalEncoder,
    NYCFeaturePipeline,
    build_and_save_feature_pipeline
)

## 1. Load Preprocessed Training Data

In [2]:
train_df = pd.read_parquet(TRAIN_CLEANED_PATH)
print(f"Loaded clean train split: {len(train_df):,} rows")
X_train = train_df[ALLOWED_FEATURES].copy()
y_train = train_df[["fare_amount", "duration_minutes"]].copy()
display(X_train.head())

Loaded clean train split: 2,402,868 rows


,tpep_pickup_datetime,PULocationID,DOLocationID,passenger_count,RatecodeID,trip_distance,VendorID
0,2022-05-01 00:00:36,246,151,1.0,1.0,4.10,1
1,2022-05-01 00:27:44,238,74,1.0,1.0,2.30,1
2,2022-05-01 00:59:00,163,260,1.0,1.0,4.20,1
3,2022-05-01 00:28:26,238,75,1.0,1.0,1.60,1
4,2022-05-01 00:07:11,164,112,1.0,1.0,3.35,2


## 2. Demonstrate Temporal & Cyclical Feature Extraction
Extracts calendar components (`pickup_hour`, `pickup_dayofweek`, `pickup_day`), sine/cosine cyclical transformations (`sin_hour`, `cos_hour`, `sin_dayofweek`, `cos_dayofweek`), weekend/rush-hour flags, and US holiday indicators.

In [3]:
temporal_extractor = TemporalFeatureExtractor()
df_temporal = temporal_extractor.transform(X_train.head(10))
display(df_temporal[["tpep_pickup_datetime", "pickup_hour", "pickup_dayofweek", "sin_hour", "cos_hour", "is_weekend", "is_rush_hour", "is_holiday"]])

,tpep_pickup_datetime,pickup_hour,pickup_dayofweek,sin_hour,cos_hour,is_weekend,is_rush_hour,is_holiday
0,2022-05-01 00:00:36,0,6,0.0,1.0,1,0,0
1,2022-05-01 00:27:44,0,6,0.0,1.0,1,0,0
2,2022-05-01 00:59:00,0,6,0.0,1.0,1,0,0
3,2022-05-01 00:28:26,0,6,0.0,1.0,1,0,0
4,2022-05-01 00:07:11,0,6,0.0,1.0,1,0,0
5,2022-05-01 00:14:38,0,6,0.0,1.0,1,0,0
6,2022-05-01 00:36:36,0,6,0.0,1.0,1,0,0
7,2022-05-01 00:05:41,0,6,0.0,1.0,1,0,0
8,2022-05-01 00:00:08,0,6,0.0,1.0,1,0,0
9,2022-05-01 00:14:49,0,6,0.0,1.0,1,0,0


## 3. Demonstrate Spatial Zone & Distance Feature Engineering
Joins pickup and dropoff zone centroids (latitude/longitude in WGS84 EPSG:4326) and calculates great-circle Haversine distance, L1 Manhattan distance, distance ratios, and airport rate indicators (`is_jfk`, `is_newark`, `is_same_zone`).

In [ ]:
spatial_extractor = SpatialZoneFeatureExtractor()
spatial_extractor.fit(X_train)
df_spatial = spatial_extractor.transform(X_train.head(10))
display(
    df_spatial[
        [
            "PULocationID",
            "DOLocationID",
            "pu_lat",
            "pu_lon",
            "do_lat",
            "do_lon",
            "haversine_distance",
            "manhattan_distance",
            "haversine_ratio",
            "is_same_zone",
            "is_jfk",
        ]
    ]
)

,PULocationID,DOLocationID,pu_lat,pu_lon,do_lat,do_lon,haversine_distance,manhattan_distance,haversine_ratio,is_same_zone,is_jfk
0,246,151,40.753309,-74.004016,40.797962,-73.968168,3.610671,4.954159,0.880437,0,0
1,238,74,40.791705,-73.973049,40.801169,-73.937346,1.978693,2.518004,0.859927,0,0
2,163,260,40.764421,-73.977569,40.744234,-73.906307,3.982100,5.117691,0.947893,0,0
3,238,75,40.791705,-73.973049,40.790011,-73.945750,1.432809,1.542976,0.894946,0,0
4,164,112,40.748575,-73.985156,40.729506,-73.949540,2.283095,3.177766,0.681318,0,0
5,79,68,40.727620,-73.985937,40.748427,-73.999918,1.613240,2.166609,0.625044,0,0
6,68,87,40.748427,-73.999918,40.706808,-74.007496,2.902868,3.267963,0.637853,0,0
7,170,244,40.747746,-73.978492,40.841708,-73.941399,6.775988,8.421009,0.720773,0,0
8,164,43,40.748575,-73.985156,40.782477,-73.965555,2.557200,3.363617,1.419878,0,0
9,163,226,40.764421,-73.977569,40.737698,-73.924673,3.327912,4.608846,0.875536,0,0


## 4. Demonstrate Target Encoding (Leakage Control)
High-cardinality zone IDs (265 zones) are encoded using Bayesian smoothed target encoding fitted **strictly** on the training set (`X_train`, `y_train`).

In [5]:
target_encoder = TargetCategoricalEncoder(smoothing=10.0)
target_encoder.fit(df_spatial, y_train.head(10))
df_encoded = target_encoder.transform(df_spatial)
display(df_encoded[["PULocationID", "PULocationID_target_enc", "DOLocationID", "DOLocationID_target_enc"]])

,PULocationID,PULocationID_target_enc,DOLocationID,DOLocationID_target_enc
0,246,15.772727,151,15.772727
1,238,14.583333,74,15.227273
2,163,15.541667,260,15.636364
3,238,14.583333,75,14.909091
4,164,14.875000,112,15.363636
5,79,15.409091,68,15.409091
6,68,16.090909,87,16.090909
7,170,17.454545,244,17.454545
8,164,14.875000,43,15.090909
9,163,15.541667,226,15.545455


## 5. Fit & Execute Unified NYCFeaturePipeline
Fits the unified `NYCFeaturePipeline` on `X_train` and `y_train` strictly, generates 29 model-ready features, and serializes the pipeline to `models/feature_pipeline.pkl`.

In [6]:
pipeline, X_train_features = build_and_save_feature_pipeline(TRAIN_CLEANED_PATH)
print(f"Generated X_train_features shape: {X_train_features.shape}")
print(f"Total engineered features: {len(pipeline.feature_names_)}")
display(X_train_features.head())

2026-08-10 18:40:20,774 - INFO - Loading cleaned training dataset from /home/wseba/github/will-i-amv/NYC_Taxi_Fare_and_Trip_Duration/dataset/train_cleaned.parquet...
2026-08-10 18:41:03,521 - INFO - Fitting NYCFeaturePipeline on X_train and y_train strictly...
2026-08-10 18:40:22,696 - INFO - Fitted NYCFeaturePipeline: 29 engineered features produced.
2026-08-10 18:40:23,592 - INFO - Saved fitted feature pipeline to /home/wseba/github/will-i-amv/NYC_Taxi_Fare_and_Trip_Duration/models/feature_pipeline.pkl


Generated X_train_features shape: (2402868, 29)
Total engineered features: 29


,PULocationID,DOLocationID,passenger_count,RatecodeID,trip_distance,VendorID,pickup_hour,pickup_dayofweek,pickup_day,sin_hour,...,do_lon,haversine_distance,manhattan_distance,haversine_ratio,is_same_zone,is_jfk,is_newark,PULocationID_target_enc,DOLocationID_target_enc,RatecodeID_target_enc
0,246,151,1.0,1.0,4.10,1,0,6,1,0.0,...,-73.968168,3.610671,4.954159,0.880437,0,0,0,12.553600,12.490848,12.691893
1,238,74,1.0,1.0,2.30,1,0,6,1,0.0,...,-73.937346,1.978693,2.518004,0.859927,0,0,0,11.435083,13.175373,12.691893
2,163,260,1.0,1.0,4.20,1,0,6,1,0.0,...,-73.906307,3.982100,5.117691,0.947893,0,0,0,12.531113,21.064805,12.691893
3,238,75,1.0,1.0,1.60,1,0,6,1,0.0,...,-73.945750,1.432809,1.542976,0.894946,0,0,0,11.435083,11.682606,12.691893
4,164,112,1.0,1.0,3.35,2,0,6,1,0.0,...,-73.949540,2.283095,3.177766,0.681318,0,0,0,12.489728,22.842241,12.691893


## 6. Verify Deserialization & Leakage Checks

In [7]:
pipeline_path = os.path.join(MODELS_DIR, "feature_pipeline.pkl")
with open(pipeline_path, "rb") as f:
    reloaded_pipeline = pickle.load(f)

# Verify transformation on test split
test_df = pd.read_parquet(TEST_CLEANED_PATH)
X_test = test_df[ALLOWED_FEATURES].copy()
X_test_features = reloaded_pipeline.transform(X_test)

print(f"Successfully transformed X_test: {X_test_features.shape}")
for banned in BANNED_COLUMNS:
    assert banned not in X_test_features.columns
print("Leakage Check: 100% Passed. Zero banned post-trip columns in transformed feature set.")

Successfully transformed X_test: (899623, 29)
Leakage Check: 100% Passed. Zero banned post-trip columns in transformed feature set.
